In [15]:
import os
import shutil
import pandas as pd

cwd = os.getcwd()

parent_folder = f"{cwd}\\DefiVulnLabs\\src\\test"

In [3]:
cwd = os.getcwd()

parent_folder = f"{cwd}\\DefiVulnLabs\\src\\test"

def list_solidity_files(parent_folder: str):
    return [f for f in os.listdir(parent_folder) if f.endswith(".sol")]

def read_file(path: str):
    with open(path, 'r', encoding='utf-8') as file:
        return file.read()

In [5]:
vuln_sol_files = list_solidity_files(parent_folder)
len(vuln_sol_files)

224

In [7]:
vuln_sol_files_copies = []
# Assuming vuln_sol_files contains the full paths of the Solidity files
for file_name in vuln_sol_files:
    file_path = f"{parent_folder}\\{file_name}"
    folder, filename = os.path.split(file_path)
    name, ext = os.path.splitext(filename)
    new_filename = f"{name}_modified{ext}"
    new_file_path = os.path.join(folder, new_filename)
    vuln_sol_files_copies.append(new_file_path)
    
    shutil.copy(file_path, new_file_path)

print("Copies created successfully.")

Copies created successfully.


In [25]:
print(len(list_solidity_files(parent_folder)))

224


In [31]:
data = {"filename": vuln_sol_files_copies}
df = pd.DataFrame(data)
df['done'] = 0
df.to_csv("defi_vuln_labs_dataset.csv")

In [35]:
df['file'] = None
for index, row in df.iterrows():
    df.loc[index, 'file'] = row['filename'].split("\\")[-1]

In [39]:
(df['file'] == None).sum()

0

In [41]:
df

,filename,done,file
0,D:\UoM\Final_Year_Project_New\vulnerability-de...,0,ApproveScam_modified.sol
1,D:\UoM\Final_Year_Project_New\vulnerability-de...,0,Array-deletion_modified.sol
2,D:\UoM\Final_Year_Project_New\vulnerability-de...,0,Backdoor-assembly_modified.sol
3,D:\UoM\Final_Year_Project_New\vulnerability-de...,0,Bypasscontract_modified.sol
4,D:\UoM\Final_Year_Project_New\vulnerability-de...,0,DataLocation_modified.sol
5,D:\UoM\Final_Year_Project_New\vulnerability-de...,0,Delegatecall_modified.sol
6,D:\UoM\Final_Year_Project_New\vulnerability-de...,0,Dirtybytes_modified.sol
7,D:\UoM\Final_Year_Project_New\vulnerability-de...,0,Divmultiply_modified.sol
8,D:\UoM\Final_Year_Project_New\vulnerability-de...,0,DOS_modified.sol
9,D:\UoM\Final_Year_Project_New\vulnerability-de...,0,ecrecover_modified.sol


In [43]:
df.to_csv("defi_vuln_labs_dataset.csv")

# After the hand labeling of vulnerability localization

In [17]:
df = pd.read_csv("defi_vuln_labs_dataset.csv")
df

,Unnamed: 0,filename,done,file,unclear
0,0,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,ApproveScam_modified.sol,1
1,1,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Array-deletion_modified.sol,0
2,2,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Backdoor-assembly_modified.sol,0
3,3,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Bypasscontract_modified.sol,0
4,4,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,DataLocation_modified.sol,0
5,5,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Delegatecall_modified.sol,0
6,6,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Dirtybytes_modified.sol,0
7,7,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Divmultiply_modified.sol,0
8,8,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,DOS_modified.sol,0
9,9,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,ecrecover_modified.sol,0


In [19]:
(df['done'] == 1).sum()

48

In [21]:
((df['done'] == 1) & (df['unclear'] == 0)).sum()

43

In [23]:
import re

def extract_vulnerable_lines(source_code):
    match = re.search(r'@vulnerable_at_lines:\s*([-?\d,\s]+)', source_code)
    if match:
        line_numbers = set(map(int, match.group(1).split(',')))
        return line_numbers
    return set()

def read_file(path):
    with open(path, 'r', encoding='utf-8') as file:
        return file.read()

def extract_and_update_vulnerable_lines_in_df(df):
    df['source_code'] = None
    df['vuln_lines'] = None
    for index, row in df.iterrows():
        file_path = f"{parent_folder}\\{row['file']}"
        content = read_file(file_path)
        if content is not None:
            line_numbers = extract_vulnerable_lines(content)
            if len(line_numbers) > 0:
                df.loc[index, "source_code"] = content
                df.loc[index, "vuln_lines"] = str(line_numbers)
                print(f"Success: {file_path} - {str(line_numbers)}")
            else:
                print(f"Something went wrong: empty line numbers set for {file_path}")
        else:
            print(f"Something went wrong: empty source code for {file_path}")

In [25]:
extract_and_update_vulnerable_lines_in_df(df)

Something went wrong: empty line numbers set for D:\UoM\Final_Year_Project_New\vulnerability-detection-and-localization\datasets\DefiVulnLabs\src\test\ApproveScam_modified.sol
Something went wrong: empty line numbers set for D:\UoM\Final_Year_Project_New\vulnerability-detection-and-localization\datasets\DefiVulnLabs\src\test\Array-deletion_modified.sol
Something went wrong: empty line numbers set for D:\UoM\Final_Year_Project_New\vulnerability-detection-and-localization\datasets\DefiVulnLabs\src\test\Backdoor-assembly_modified.sol
Something went wrong: empty line numbers set for D:\UoM\Final_Year_Project_New\vulnerability-detection-and-localization\datasets\DefiVulnLabs\src\test\Bypasscontract_modified.sol
Something went wrong: empty line numbers set for D:\UoM\Final_Year_Project_New\vulnerability-detection-and-localization\datasets\DefiVulnLabs\src\test\DataLocation_modified.sol
Something went wrong: empty line numbers set for D:\UoM\Final_Year_Project_New\vulnerability-detection-and-

In [57]:
df

,Unnamed: 0,filename,done,file,unclear,source_code,vuln_lines
0,0,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,ApproveScam_modified.sol,1,// SPDX-License-Identifier: MIT\npragma solidi...,{72}
1,1,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Array-deletion_modified.sol,0,// SPDX-License-Identifier: MIT\npragma solidi...,{32}
2,2,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Backdoor-assembly_modified.sol,0,// SPDX-License-Identifier: MIT\npragma solidi...,"{44, 46, 47, 51, 53, 54}"
3,3,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Bypasscontract_modified.sol,0,// SPDX-License-Identifier: MIT\npragma solidi...,"{25, 26, 27, 34}"
4,4,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,DataLocation_modified.sol,0,// SPDX-License-Identifier: MIT\npragma solidi...,"{36, 37}"
5,5,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Delegatecall_modified.sol,0,// SPDX-License-Identifier: MIT\npragma solidi...,{40}
6,6,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Dirtybytes_modified.sol,0,// SPDX-License-Identifier: MIT\npragma solidi...,{35}
7,7,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,Divmultiply_modified.sol,0,// SPDX-License-Identifier: MIT\npragma solidi...,{37}
8,8,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,DOS_modified.sol,0,// SPDX-License-Identifier: MIT\npragma solidi...,"{32, 33}"
9,9,D:\UoM\Final_Year_Project_New\vulnerability-de...,1,ecrecover_modified.sol,0,// SPDX-License-Identifier: MIT\npragma solidi...,"{40, 57}"


In [59]:
df.to_csv("processed/defi_vuln_labs_dataset_with_vuln_line_numbers.csv")